# CRbot: PPO self-play from crbot_il_v2 (Colab T4)

Fine-tunes the imitation-learning policy `crbot_il_v2` with PPO against a fictitious self-play pool
(ScriptedBot 30%, frozen crbot_il_v2 30%, earlier PPO snapshots 40%) on the CRbot engine, with a
KL penalty that keeps it close to the human-like IL policy. Code: `backend_engine/brain/ppo_trainer.py`.

**Deck specialist (Hog 2.6):** start the notebook with a cell
`%env PPO_NAME=crbot_ppo_hog26`, `%env PPO_INIT=crbot_il_v2_hog26`, `%env PPO_DECKS=Hog 2.6 Cycle`, `%env PPO_HOURS=12`
and pack the bundle with `--il-model crbot_il_v2_hog26`. Checkpoints then go to `checkpoints/crbot_ppo_hog26/`.

**Before you run:**
1. On the Mac: `.venv/bin/python tools/pack_ppo_bundle.py` → `dist/crbot_ppo_bundle.zip` (~11 MB).
2. Upload it to Google Drive as `MyDrive/CRbot_Training/crbot_ppo_bundle.zip`.
3. Runtime → Change runtime type → **T4 GPU**, then Run all.

Checkpoints go to `MyDrive/CRbot_Training/checkpoints/ppo/` every few updates; after a disconnect just run all
again and it resumes. The best model is exported to `MyDrive/CRbot_Training/models/crbot_ppo_v2/`
(`crbot_ppo_v2.onnx` + `.npz` + `.json`); put the files in `CR-Pro-Coach/pretrained_weights/crbot_ppo_v2/` on the Mac
and pick the **Neural (PPO v2)** persona, then calibrate with `cloud_training/gauntlet_eval.py`.

In [ ]:
# 1. Environment: Drive, bundle, packages
import os, sys, json, time, zipfile, shutil
IN_COLAB = "google.colab" in sys.modules
# Run options (environment variables, e.g. first cells `%env PPO_NAME=crbot_ppo_hog26` ...):
#   PPO_NAME   exported model + its own checkpoint folder (default crbot_ppo_v2)
#   PPO_INIT   IL model inside the bundle to start from (default crbot_il_v2; deck specialist: crbot_il_v2_hog26)
#   PPO_DECKS  preset names the learner plays, comma-separated (e.g. "Hog 2.6 Cycle"); empty = every preset
#   PPO_HOURS  stop cleanly after this many hours (e.g. 12 per day; the next run resumes)
#   PPO_UPDATES, PPO_ENVS, PPO_MINIBATCH, PPO_CKPT_EVERY, PPO_MAX_MS
PPO_NAME = os.environ.get("PPO_NAME", "crbot_ppo_v2")
PPO_INIT = os.environ.get("PPO_INIT", "crbot_il_v2")
PPO_DECKS = tuple(d.strip() for d in os.environ.get("PPO_DECKS", "").split(",") if d.strip())
PPO_HOURS = float(os.environ["PPO_HOURS"]) if os.environ.get("PPO_HOURS") else None
if IN_COLAB:
    !nvidia-smi -L
    !pip -q install onnx onnxruntime onnxscript
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/CRbot_Training"
    WORK = "/content/crbot"
else:                                         # local smoke test: PPO_ROOT (bundle + outputs), PPO_WORK
    ROOT = os.environ.get("PPO_ROOT", "ppo_root")
    WORK = os.environ.get("PPO_WORK", "ppo_work")
BUNDLE = f"{ROOT}/crbot_ppo_bundle.zip"
# the original run keeps its old checkpoint folder; every other model name gets its own
CKPT_DIR = f"{ROOT}/checkpoints/ppo" if PPO_NAME == "crbot_ppo_v2" else f"{ROOT}/checkpoints/{PPO_NAME}"
OUT_DIR = f"{ROOT}/models/{PPO_NAME}"
assert os.path.exists(BUNDLE), f"upload the bundle to {BUNDLE} first (tools/pack_ppo_bundle.py)"
shutil.rmtree(WORK, ignore_errors=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall(WORK)
manifest = json.load(open(f"{WORK}/BUNDLE_MANIFEST.json"))
print(f"bundle {manifest['git_commit']} ({len(manifest['files'])} files, created {manifest['created']})")
sys.path.insert(0, WORK)
os.chdir(WORK)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(max(1, os.cpu_count() or 1))
print("device", DEVICE, "| torch", torch.__version__, "| CPUs", os.cpu_count())

In [ ]:
# 2. Settings (defaults: ~15-25 s per update on a T4 runtime, ~300 updates in 1.5-2 h)
from backend_engine.brain.ppo_trainer import PPOConfig, PPOTrainer
CFG = PPOConfig(
    n_envs=int(os.environ.get("PPO_ENVS", 8)),           # matches per update
    updates=int(os.environ.get("PPO_UPDATES", 300)),
    max_match_ms=int(os.environ["PPO_MAX_MS"]) if os.environ.get("PPO_MAX_MS") else None,
    lr=2e-5, clip=0.2, epochs=4, minibatch=int(os.environ.get("PPO_MINIBATCH", 1024)),
    gamma=0.995, lam=0.95, ent_coef=0.003, vf_coef=0.5, kl_coef=0.05,
    w_tower=0.5, w_crown=0.5, leak_penalty=0.01,
    opp_mix=(0.3, 0.3, 0.4), snapshot_every=5, pool_size=10, ckpt_every=int(os.environ.get("PPO_CKPT_EVERY", 5)),
    score_window=5, device=DEVICE, seed=0,
    learner_decks=PPO_DECKS, time_budget_h=PPO_HOURS, export_name=PPO_NAME,
)
print(CFG)

In [ ]:
# 3. Trainer: initialised from PPO_INIT (its .npz weights, checked against the ONNX file; also the KL reference);
#    resumes automatically from CKPT_DIR
trainer = PPOTrainer(CFG, il_onnx=f"{WORK}/pretrained_weights/{PPO_INIT}/{PPO_INIT}.onnx",
                     ckpt_dir=CKPT_DIR, out_dir=OUT_DIR)
print(f"starting at update {trainer.update} / {CFG.updates}, {len(trainer.snapshots)} snapshots in the pool")

In [ ]:
# 4. Train (safe to interrupt: the last checkpoint is at most `ckpt_every` updates old)
t0 = time.time()
summary = trainer.train()
print(summary, f"{(time.time() - t0) / 60:.1f} min")

In [ ]:
# 5. Learning curve (score = (W + D/2) / N per update; 0.5 = even) and per-opponent results
import numpy as np
h = trainer.history
if h:
    s = np.array([r["score"] for r in h])
    k = min(10, len(s))
    smooth = np.convolve(s, np.ones(k) / k, mode="valid")
    print("score, moving average of", k, "updates:", np.round(smooth[-20:], 3))
    for opp in ("scripted", "il", "snapshot"):
        v = [r["by_opponent"][opp] for r in h[-20:] if opp in r["by_opponent"]]
        if v:
            print(f"last 20 updates vs {opp:9s}: {np.mean(v):.3f} over {len(v)} updates")
    print("last kl_ref", h[-1].get("kl_ref"), "| approx_kl", h[-1].get("approx_kl"), "| clipfrac", h[-1].get("clipfrac"))

In [ ]:
# 6. Check the exported model the way the app loads it (onnxruntime, NeuralBot contract) and download it
from backend_engine.brain.neural_bot import load_neural_bot
from backend_engine.simulator.card_db import CardDB
bot = load_neural_bot(CardDB.load(), 1, folder=OUT_DIR, name=PPO_NAME)
print("NeuralBot loads", bot.meta["name"], "| gate", bot.gate_threshold, "| every", bot.decide_every_ms, "ms",
      "| best rolling score", bot.meta["ppo"]["best_rolling_score"])
if IN_COLAB:
    from google.colab import files
    pack = f"/content/{PPO_NAME}_pack"
    os.makedirs(pack, exist_ok=True)
    for name in (f"{PPO_NAME}.onnx", f"{PPO_NAME}.npz", f"{PPO_NAME}.json"):
        shutil.copy(f"{OUT_DIR}/{name}", pack)
    files.download(shutil.make_archive(f"/content/{PPO_NAME}", "zip", pack))